# Colab: Gemma-4 E4B Vision fine-tune on WLASL Top-50 video frames

This notebook fine-tunes `unsloth/gemma-4-E4B-it` with Unsloth Vision on extracted WLASL Top-50 frame sequences.

Input contract from `scripts/data/extract_wlasl_video_frames.py`:

```text
data/video_finetune/top50/
  frames/{gloss}/{video_id}/frame_000.jpg ... frame_019.jpg
  train.jsonl
  val.jsonl
  test.jsonl
  labels.txt
```

Each manifest row contains `sample_id`, `gloss`, `split`, and exactly 30 ordered `frame_paths`.

**Recommended first goal:** prove the pipeline with a short smoke train, then increase steps/epochs only after one generation/eval path works.


In [17]:
%%capture
# Colab/Unsloth install. Restart runtime if Colab asks, then run from this cell again.
!pip install -U --no-cache-dir unsloth unsloth_zoo datasets huggingface_hub accelerate bitsandbytes trl
!pip install --force-reinstall --no-cache-dir "pillow==11.3.0"
!pip install --no-deps --upgrade timm


In [18]:
# HF / Unsloth stability preflight. Run BEFORE importing unsloth.
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("UNSLOTH_DISABLE_STATISTICS=", os.environ.get("UNSLOTH_DISABLE_STATISTICS"))


UNSLOTH_DISABLE_STATISTICS= 1


In [19]:
from pathlib import Path
import json, random, re, shutil, zipfile, os
from collections import Counter
from getpass import getpass

import torch
from PIL import Image
from datasets import Dataset
from huggingface_hub import login, HfApi

SEED = 3407
random.seed(SEED)

HF_NAMESPACE = "AlexD281"
BASE_MODEL = "unsloth/gemma-4-E4B-it"
RUN_NAME = "asl-gemma4-e4b-video-top50-lora"
ADAPTER_REPO_ID = f"{HF_NAMESPACE}/{RUN_NAME}"

WORK_DIR = Path("/content/asl_gemma4_e4b_video_top50")
DATASET_DIR = WORK_DIR / "data" / "video_finetune" / "top50"
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME
PRED_DIR = WORK_DIR / "predictions"
for p in [DATASET_DIR, OUTPUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

NUM_FRAMES = 30
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 8
MAX_LENGTH = 20_000
MAX_SEQ_LENGTH = MAX_LENGTH
LORA_RANK = 16
LORA_ALPHA = 16

# Start tiny; once the forward/generation/eval path works, set SMOKE_MAX_STEPS=None and NUM_TRAIN_EPOCHS=1..3.
SMOKE_MAX_STEPS = None
NUM_TRAIN_EPOCHS = 2

print("Base model:", BASE_MODEL)
print("Adapter repo:", ADAPTER_REPO_ID)
print("Dataset dir:", DATASET_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")


Base model: unsloth/gemma-4-E4B-it
Adapter repo: AlexD281/asl-gemma4-e4b-video-top50-lora
Dataset dir: /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Provide the Top-50 frame dataset

Use one of these simple paths:

1. Upload a zip containing `top50/` or the contents of `data/video_finetune/top50/`, then set `USE_UPLOAD = True`.
2. Mount Google Drive and point `DRIVE_TOP50_DIR` to the extracted `top50` folder.

The final `DATASET_DIR` must contain `train.jsonl`, `val.jsonl`, `test.jsonl`, `labels.txt`, and `frames/`.


In [20]:
# Preferred Colab path: keep ONE zip on Google Drive, copy that zip to /content, then unzip locally.
USE_DRIVE_ZIP = True
USE_DRIVE_FOLDER = False
USE_UPLOAD = False

DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/asl/top50_video_finetune_30frames.zip")
DRIVE_TOP50_DIR = Path("/content/drive/MyDrive/asl/video_finetune/top50")

from google.colab import drive

def find_top50_source(extract_root: Path) -> Path:
    candidates = [
        extract_root / "top50",
        extract_root / "data" / "video_finetune" / "top50",
        extract_root,
    ]
    source = next((p for p in candidates if (p / "train.jsonl").exists() and (p / "frames").exists()), None)
    if source is None:
        raise FileNotFoundError(f"Could not find top50 dataset inside {extract_root}")
    return source

if sum([USE_DRIVE_ZIP, USE_DRIVE_FOLDER, USE_UPLOAD]) != 1:
    raise ValueError("Choose exactly one dataset input mode: USE_DRIVE_ZIP, USE_DRIVE_FOLDER, or USE_UPLOAD")

if USE_DRIVE_ZIP:
    drive.mount('/content/drive')
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(f"Drive zip not found: {DRIVE_ZIP_PATH}")
    local_zip = WORK_DIR / DRIVE_ZIP_PATH.name
    print("Copying one zip from Drive to local Colab disk...", DRIVE_ZIP_PATH)
    shutil.copy2(DRIVE_ZIP_PATH, local_zip)
    print("Unzipping locally...", local_zip)
    extract_root = WORK_DIR / "drive_zip_dataset"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_root)
    source = find_top50_source(extract_root)
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(source, DATASET_DIR)
    print("Prepared dataset at", DATASET_DIR)

if USE_DRIVE_FOLDER:
    drive.mount('/content/drive')
    if not DRIVE_TOP50_DIR.exists():
        raise FileNotFoundError(f"Drive dataset folder not found: {DRIVE_TOP50_DIR}")
    print("WARNING: copying a Drive folder with thousands of JPGs can take 10+ minutes.")
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(DRIVE_TOP50_DIR, DATASET_DIR)
    print("Copied Drive folder dataset to", DATASET_DIR)

if USE_UPLOAD:
    from google.colab import files
    print("Upload a .zip containing the top50 dataset folder or its contents.")
    uploaded = files.upload()
    zip_paths = [Path(name) for name in uploaded if name.endswith(".zip")]
    if not zip_paths:
        raise FileNotFoundError("Upload a .zip file for the top50 dataset")
    zip_path = zip_paths[0]
    extract_root = WORK_DIR / "uploaded_dataset"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_root)
    source = find_top50_source(extract_root)
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(source, DATASET_DIR)
    print("Copied uploaded dataset to", DATASET_DIR)

required = ["train.jsonl", "val.jsonl", "test.jsonl", "labels.txt", "frames"]
for name in required:
    path = DATASET_DIR / name
    print(name, "OK" if path.exists() else "MISSING", path)
    if not path.exists():
        raise FileNotFoundError(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying one zip from Drive to local Colab disk... /content/drive/MyDrive/asl/top50_video_finetune_30frames.zip
Unzipping locally... /content/asl_gemma4_e4b_video_top50/top50_video_finetune_30frames.zip
Prepared dataset at /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50
train.jsonl OK /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50/train.jsonl
val.jsonl OK /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50/val.jsonl
test.jsonl OK /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50/test.jsonl
labels.txt OK /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50/labels.txt
frames OK /content/asl_gemma4_e4b_video_top50/data/video_finetune/top50/frames


In [21]:
def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

labels = [line.strip() for line in (DATASET_DIR / "labels.txt").read_text().splitlines() if line.strip()]
label_set = set(labels)
train_rows = read_jsonl(DATASET_DIR / "train.jsonl")
val_rows = read_jsonl(DATASET_DIR / "val.jsonl")
test_rows = read_jsonl(DATASET_DIR / "test.jsonl")

print("labels", len(labels), labels[:10])
for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    print(name, len(rows), Counter(row["gloss"] for row in rows).most_common(5))
    bad = [row for row in rows if row["gloss"] not in label_set or len(row.get("frame_paths", [])) != NUM_FRAMES]
    if bad:
        raise ValueError(f"{name} has invalid rows; first={bad[0]}")
print("Dataset manifest validation passed")


labels 50 ['drink', 'like', 'wrong', 'forget', 'computer', 'finish', 'hot', 'mother', 'now', 'orange']
train 183 [('like', 6), ('now', 6), ('book', 5), ('drink', 5), ('before', 5)]
val 59 [('all', 3), ('kiss', 3), ('forget', 3), ('drink', 2), ('computer', 2)]
test 63 [('black', 3), ('hot', 3), ('many', 3), ('chair', 2), ('yes', 2)]
Dataset manifest validation passed


## Tiny overfit diagnostic

Use this mode to verify the model can memorize a tiny 20-example training subset before spending time on full Top-50 runs.


In [ ]:
# Recommendation 1: tiny overfit diagnostic mode.
# Turn this ON to check whether the model can memorize 20 training examples.
# Run this cell before the dataset construction cell below.
OVERFIT_DIAGNOSTIC = True

if OVERFIT_DIAGNOSTIC:
    TRAIN_LIMIT = 20
    VAL_LIMIT = 20
    SMOKE_MAX_STEPS = None
    NUM_TRAIN_EPOCHS = 10
    print("OVERFIT_DIAGNOSTIC enabled")
    print("Training on 20 examples for 10 epochs; evaluate on those same examples after training.")
else:
    TRAIN_LIMIT = None
    VAL_LIMIT = None
    # Keep whatever full-run values were set in the main config cell.
    print("OVERFIT_DIAGNOSTIC disabled; using full dataset limits.")


In [22]:
TOP50_ALLOWLIST_TEXT = "\n".join(labels)
INSTRUCTION_TOP50 = f"""You are given 30 ordered frames from one isolated ASL video.

Task:
Classify the ASL sign being performed.

Rules:
- Answer with exactly one label.
- Do not explain.
- Do not include punctuation.
- Do not repeat the allowed labels.
- The answer must be one of the allowed labels below.

Allowed labels:
{TOP50_ALLOWLIST_TEXT}

Answer:"""

def clean_label(label: str) -> str:
    return str(label).strip().lower()

def load_images(row, dataset_dir=DATASET_DIR):
    images = []
    for rel_path in row["frame_paths"]:
        image_path = dataset_dir / rel_path
        if not image_path.exists():
            raise FileNotFoundError(image_path)
        images.append(Image.open(image_path).convert("RGB"))
    return images

def row_to_messages(row):
    images = load_images(row)
    answer = clean_label(row["gloss"])
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    *[{"type": "image", "image": img} for img in images],
                    {"type": "text", "text": INSTRUCTION_TOP50},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ],
        "sample_id": row["sample_id"],
        "gloss": answer,
    }

# Optional smoke/diagnostic subset keeps Colab iteration fast. Set to None for the full split.
TRAIN_LIMIT = globals().get("TRAIN_LIMIT", None)
VAL_LIMIT = globals().get("VAL_LIMIT", None)
train_dataset = [row_to_messages(r) for r in train_rows[:TRAIN_LIMIT]]
val_dataset = [row_to_messages(r) for r in val_rows[:VAL_LIMIT]]
print("train samples", len(train_dataset), "val samples", len(val_dataset))
print(train_dataset[0]["sample_id"], train_dataset[0]["gloss"], len(train_dataset[0]["messages"][0]["content"]))



train samples 183 val samples 59
book_68011 book 31


In [23]:
from unsloth import FastVisionModel

try:
    model, processor = FastVisionModel.from_pretrained(
        BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
except TimeoutError as exc:
    raise RuntimeError(
        "Unsloth timed out before model loading. Restart runtime and rerun the preflight cell "
        "that sets UNSLOTH_DISABLE_STATISTICS=1 before importing Unsloth."
    ) from exc


model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear",
)
print("Loaded", BASE_MODEL)
processor.tokenizer.model_max_length = MAX_LENGTH
processor.tokenizer.init_kwargs["model_max_length"] = MAX_LENGTH
if hasattr(model, "config"):
    model.config.max_position_embeddings = max(getattr(model.config, "max_position_embeddings", 0) or 0, MAX_LENGTH)
print("Processor:", type(processor).__name__)
print("Tokenizer model_max_length:", processor.tokenizer.model_max_length)




==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Loaded unsloth/gemma-4-E4B-it
Processor: Gemma4Processor
Tokenizer model_max_length: 20000


In [24]:
# One-sample processor/generation sanity check before training.
FastVisionModel.for_inference(model)
row = test_rows[0]
images = load_images(row)
messages = [
    {
        "role": "user",
        "content": [
            *[{"type": "image"} for _ in images],
            {"type": "text", "text": INSTRUCTION_TOP50},
        ],
    }
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    images=images,
    text=input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, do_sample=False)
raw = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("gold:", row["gloss"])
print("raw generation:\n", raw[-500:])
FastVisionModel.for_training(model)


gold: drink
raw generation:
 
Rules:
- Answer with exactly one label.
- Do not explain.
- Do not include punctuation.
- Do not repeat the allowed labels.
- The answer must be one of the allowed labels below.

Allowed labels:
drink
like
wrong
forget
computer
finish
hot
mother
now
orange
hearing
color
birthday
need
book
before
deaf
fine
no
yes
all
black
kiss
study
white
dance
but
cook
paper
visit
door
college
your
use
better
last year
will
chair
clothes
candy
year
many
woman
blue
fish
hat
bird
cow
enjoy
meet

Answer:
model
no


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0): Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, ou

In [25]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

sft_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    max_grad_norm=0.3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="steps" if SMOKE_MAX_STEPS else "epoch",
    eval_steps=10 if SMOKE_MAX_STEPS else None,
    save_strategy="steps" if SMOKE_MAX_STEPS else "epoch",
    save_steps=10 if SMOKE_MAX_STEPS else None,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=MAX_LENGTH,  # Must be > 7,680 image tokens; avoids image-token truncation.
    max_seq_length=MAX_SEQ_LENGTH
)
if SMOKE_MAX_STEPS:
    sft_kwargs["max_steps"] = SMOKE_MAX_STEPS
else:
    sft_kwargs["num_train_epochs"] = NUM_TRAIN_EPOCHS

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(**sft_kwargs),
)
trainer



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [26]:
trainer_stats = trainer.train()

print("train_dataset size:", len(train_dataset))
print("val_dataset size:", len(val_dataset))
print("trainer max steps:", trainer.state.max_steps)
print("trainer global step:", trainer.state.global_step)

print("\nTRAINING LOG HISTORY")
for item in trainer.state.log_history:
    print(item)

trainer_stats



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 183 | Num Epochs = 2 | Total steps = 92
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.327526,7.280239
2,0.132562,7.256680


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/asl_gemma4_e4b_video_top50/outputs/asl-gemma4-e4b-video-top50-lora/checkpoint-46/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/asl_gemma4_e4b_video_top50/outputs/asl-gemma4-e4b-video-top50-lora/checkpoint-92/tokenizer_config.json.


TrainOutput(global_step=92, training_loss=0.7881905783129775, metrics={'train_runtime': 1970.5628, 'train_samples_per_second': 0.186, 'train_steps_per_second': 0.047, 'total_flos': 7.924570842791232e+16, 'train_loss': 0.7881905783129775, 'epoch': 2.0})

In [27]:
# Save local LoRA adapter and processor/tokenizer.
ADAPTER_DIR = OUTPUT_DIR / "adapter"
model.save_pretrained(str(ADAPTER_DIR))
processor.tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapter:", ADAPTER_DIR)
print(sorted(p.name for p in ADAPTER_DIR.iterdir()))


Unsloth: Restored added_tokens_decoder metadata in /content/asl_gemma4_e4b_video_top50/outputs/asl-gemma4-e4b-video-top50-lora/adapter/tokenizer_config.json.


saved adapter: /content/asl_gemma4_e4b_video_top50/outputs/asl-gemma4-e4b-video-top50-lora/adapter
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [28]:
def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def extract_predicted_label(raw: str, label_set: set[str]) -> str:
    text = normalize_text(raw)

    # If the model includes the prompt or a formatted response, keep only the answer span.
    if "answer:" in text:
        text = text.split("answer:")[-1].strip()

    # Remove common assistant prefixes.
    text = re.sub(r"^(assistant|model|response)\s*:\s*", "", text).strip()

    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Prefer an exact label line, inspecting from the bottom in case of preamble text.
    for line in reversed(lines):
        cleaned = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", line.strip().lower())
        if cleaned in label_set:
            return cleaned

    # Try the first short chunk. Example: "drink." -> "drink".
    first_chunk = re.split(r"[\n,.;]", text.strip())[0].strip()
    first_chunk = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", first_chunk)
    if first_chunk in label_set:
        return first_chunk

    # Fall back to a unique valid label mention. Multiple labels usually means prompt echo.
    found = []
    for label in sorted(label_set, key=len, reverse=True):
        pattern = r"(?<![a-z0-9])" + re.escape(label) + r"(?![a-z0-9])"
        if re.search(pattern, text):
            found.append(label)

    unique_found = list(dict.fromkeys(found))
    if len(unique_found) == 1:
        return unique_found[0]

    return "__invalid__"

def predict_row(row):
    FastVisionModel.for_inference(model)
    images = load_images(row)
    messages = [
        {
            "role": "user",
            "content": [
                *[{"type": "image"} for _ in images],
                {"type": "text", "text": INSTRUCTION_TOP50},
            ],
        }
    ]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        images=images,
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            do_sample=False,
        )

    # Decode only newly generated tokens so prompt echo cannot poison scoring.
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)
    pred = extract_predicted_label(raw, label_set)
    return pred, raw

EVAL_LIMIT = len(test_rows)
preds = []
for row in test_rows[:EVAL_LIMIT]:
    pred, raw = predict_row(row)
    preds.append({
        "sample_id": row["sample_id"],
        "gold": clean_label(row["gloss"]),
        "pred": pred,
        "correct": pred == clean_label(row["gloss"]),
        "raw": raw,
    })

acc = sum(p["correct"] for p in preds) / len(preds) if preds else 0.0
print(f"eval_n={len(preds)} accuracy={acc:.3f}")
print(preds[:5])
(PRED_DIR / "top50_predictions.jsonl").write_text("".join(json.dumps(p) + "\n" for p in preds))
print("wrote", PRED_DIR / "top50_predictions.jsonl")

print("\nRAW OUTPUT SAMPLES")
for p in preds[:20]:
    print("=" * 80)
    print("sample_id:", p["sample_id"])
    print("gold:", p["gold"])
    print("pred:", p["pred"])
    print("correct:", p["correct"])
    print("raw:")
    print(p["raw"])


eval_n=63 accuracy=0.063
[{'sample_id': 'drink_17730', 'gold': 'drink', 'pred': 'drink', 'correct': True, 'raw': 'drink'}, {'sample_id': 'computer_12306', 'gold': 'computer', 'pred': 'drink', 'correct': False, 'raw': 'drink'}, {'sample_id': 'chair_09847', 'gold': 'chair', 'pred': 'drink', 'correct': False, 'raw': 'drink'}, {'sample_id': 'chair_09855', 'gold': 'chair', 'pred': 'drink', 'correct': False, 'raw': 'drink'}, {'sample_id': 'clothes_11305', 'gold': 'clothes', 'pred': 'drink', 'correct': False, 'raw': 'drink'}]
wrote /content/asl_gemma4_e4b_video_top50/predictions/top50_predictions.jsonl

RAW OUTPUT SAMPLES
sample_id: drink_17730
gold: drink
pred: drink
correct: True
raw:
drink
sample_id: computer_12306
gold: computer
pred: drink
correct: False
raw:
drink
sample_id: chair_09847
gold: chair
pred: drink
correct: False
raw:
drink
sample_id: chair_09855
gold: chair
pred: drink
correct: False
raw:
drink
sample_id: clothes_11305
gold: clothes
pred: drink
correct: False
raw:
drink
sam

## Tiny overfit diagnostic evaluation

After training, run this cell to see whether the model can memorize the same 20 examples it trained on.


In [ ]:
# Recommendation 1 follow-up: evaluate the trained model on the same tiny train subset.
# Use this after trainer.train() and after the predict_row/extraction functions are defined.
def evaluate_rows(rows, name="eval", limit=None):
    selected_rows = rows[:limit] if limit else rows
    preds_local = []

    for row in selected_rows:
        pred, raw = predict_row(row)
        gold = clean_label(row["gloss"])
        preds_local.append({
            "sample_id": row["sample_id"],
            "gold": gold,
            "pred": pred,
            "correct": pred == gold,
            "raw": raw,
        })

    acc = sum(p["correct"] for p in preds_local) / len(preds_local) if preds_local else 0.0
    pred_counts = Counter(p["pred"] for p in preds_local)

    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"examples={len(preds_local)} accuracy={acc:.3f}")
    print("top predictions:")
    for label, count in pred_counts.most_common(20):
        print(f"  {label}: {count}")

    print("\nwrong examples:")
    for p in preds_local:
        if not p["correct"]:
            print(f"  {p['sample_id']}: gold={p['gold']} pred={p['pred']} raw={p['raw']!r}")

    return preds_local

if globals().get("OVERFIT_DIAGNOSTIC", False):
    overfit_train_preds = evaluate_rows(train_rows, name="overfit_train_subset", limit=TRAIN_LIMIT)
else:
    print("OVERFIT_DIAGNOSTIC is off; skipping tiny-overfit train-subset eval.")


In [29]:
import json
from pathlib import Path
from collections import Counter
predictions_path = Path("/content/asl_gemma4_e4b_video_top50/predictions/top50_predictions.jsonl")
rows = [json.loads(line) for line in predictions_path.read_text().splitlines() if line.strip()]
n = len(rows)
correct = sum(row["correct"] for row in rows)
acc = correct / n if n else 0
pred_counts = Counter(row["pred"] for row in rows)
gold_counts = Counter(row["gold"] for row in rows)
confusions = Counter((row["gold"], row["pred"]) for row in rows if not row["correct"])
print(f"Examples: {n}")
print(f"Correct: {correct}")
print(f"Accuracy: {acc:.2%}")
print(f"Random Top-50 baseline: {1/50:.2%}")
print("\nTop predictions:")
for label, count in pred_counts.most_common(10):
    print(f"  {label}: {count}")
print("\nGold labels in eval:")
for label, count in gold_counts.most_common(10):
    print(f"  {label}: {count}")
print("\nMost common mistakes:")
for (gold, pred), count in confusions.most_common(15):
    print(f"  {gold} -> {pred}: {count}")
print("\nCorrect samples:")
for row in rows:
    if row["correct"]:
        print(f"  {row['sample_id']}: {row['gold']}")

Examples: 63
Correct: 4
Accuracy: 6.35%
Random Top-50 baseline: 2.00%

Top predictions:
  drink: 28
  yes: 15
  hearing: 8
  no: 6
  kiss: 2
  wrong: 1
  fine: 1
  need: 1
  deaf: 1

Gold labels in eval:
  black: 3
  hot: 3
  many: 3
  chair: 2
  yes: 2
  finish: 2
  mother: 2
  orange: 2
  woman: 2
  fish: 2

Most common mistakes:
  chair -> drink: 2
  black -> hearing: 2
  hot -> drink: 2
  many -> yes: 2
  orange -> drink: 2
  woman -> drink: 2
  fish -> yes: 2
  cook -> drink: 2
  better -> yes: 2
  computer -> drink: 1
  clothes -> drink: 1
  deaf -> hearing: 1
  fine -> yes: 1
  no -> drink: 1
  year -> drink: 1

Correct samples:
  drink_17730: drink
  yes_64275: yes
  yes_64297: yes
  kiss_31758: kiss


In [30]:
# Download predictions JSONL to your local machine.
from google.colab import files

predictions_path = Path("/content/asl_gemma4_e4b_video_top50/predictions/top50_predictions.jsonl")
if not predictions_path.exists():
    raise FileNotFoundError(
        f"Predictions file not found: {predictions_path}. Run the evaluation/prediction cell first."
    )
print("Downloading", predictions_path, "size_bytes=", predictions_path.stat().st_size)
files.download(str(predictions_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# Optional: push LoRA adapter to Hugging Face.
PUSH_TO_HUB = False
if PUSH_TO_HUB:
    hf_token = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
    login(token=hf_token, add_to_git_credential=False)
    api = HfApi(token=hf_token)
    api.create_repo(ADAPTER_REPO_ID, repo_type="model", private=True, exist_ok=True)
    model.push_to_hub(ADAPTER_REPO_ID, token=hf_token)
    processor.tokenizer.push_to_hub(ADAPTER_REPO_ID, token=hf_token)
    print("pushed adapter to", ADAPTER_REPO_ID)
else:
    print("PUSH_TO_HUB=False; adapter remains local at", ADAPTER_DIR)


PUSH_TO_HUB=False; adapter remains local at /content/asl_gemma4_e4b_video_top50/outputs/asl-gemma4-e4b-video-top50-lora/adapter


## If Colab OOMs

Change the local frame extraction settings and regenerate the dataset in this order:

1. 30 frames @ 448
2. 20 frames @ 448
3. 16 frames @ 448
4. 12 frames @ 448
5. 8-12 frames @ 336/384

Then update `NUM_FRAMES` and `IMAGE_SIZE` in this notebook to match the regenerated manifest.
